# Classifying BBQ product reviews with Jev

Fictional customer reviews of barbecue products, broken into sentences, with every sentence classified by
[TypeSafe AI's Jev](https://docs.typesafe.ai/introduction) - a *System One* model that answers typed questions
(`Choice`, `Score`, `Noul`) with probabilities and a confidence, instead of generating text to be parsed.

1. Load the raw JSON into a flat Polars frame.
2. Break each review into sentences.
3. Ask Jev about every sentence, with the whole review as context.
4. Roll the answers up into things a product or support team can act on.

All the logic lives in `src/` (`review_wrangler`, `jev_classifier`, `review_insights`); this notebook only runs it.
`TYPESAFE_API_KEY` is read from `.env`.

In [1]:
from pathlib import Path

import polars as pl
from dotenv import load_dotenv

from jev_classifier import FRUSTRATION_LEVELS, KEY, PROBLEM_CATEGORIES, QUESTION_SET_VERSION, build_questions, classify_sentences, transcript
from review_insights import cross_mentions, flagged, frustration_by_rating, needs_review, problems_by_product, review_rollup
from review_wrangler import load_reviews, product_catalogue, split_sentences

load_dotenv(Path.cwd().parent / ".env")
OUTPUT = Path.cwd().parent / "data" / "output"
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_cols(16);

## 1. Load the reviews

One row per review, with the nested `demographics` object flattened into columns. `review_key` hashes product + text: the sample repeats the same text under different customers, and classification is keyed on it so each distinct text is asked about once.

In [2]:
reviews = load_reviews()
reviews_df = reviews.collect()
print(reviews_df.shape, "-", reviews_df["review_key"].n_unique(), "distinct texts")
reviews_df.head()

(256, 11) - 168 distinct texts


review_row,review_key,rating,review_text,product_name,product_category,gender,age,country,days_as_customer,lifetime_revenue
u32,str,i64,str,str,str,str,i64,str,i64,f64
0,"""dd3ea70e9f3a""",5,"""Absolutely blown away by the FireMaster Pro 3000! Been grilling for over 20 years and this…","""FireMaster Pro 3000 Gas Grill""","""grills""","""male""",45,"""United States""",89,1299.99
1,"""01bf8e70b1ca""",2,"""Really disappointed with these tongs. Handle broke after just 3 uses. Cheap plastic that c…","""GrillMaster Elite Tongs""","""accessories""","""female""",32,"""Canada""",12,24.99
2,"""737aa10e04fa""",4,"""This charcoal lights quickly and burns clean with minimal ash. Good heat output and lasts …","""SmokeRing Premium Lump Charcoal""","""consumables""","""male""",38,"""Australia""",184,156.47
3,"""452620b29c53""",1,"""Total waste of money! This grill arrived with multiple dents and scratches, clearly damage…","""BackYard King Compact Grill""","""grills""","""female""",28,"""United Kingdom""",21,189.99
4,"""4b764b6fe747""",5,"""Perfect for quick starts! Lights charcoal every time without that awful chemical smell oth…","""FlameStarter Natural Fire Lighter""","""consumables""","""male""",52,"""Germany""",412,83.45


The **master product list** - the options Jev picks mentions from:

In [3]:
catalogue = product_catalogue(reviews)
catalogue

product_name,product_category
str,str
"""ChefsPride Stainless Steel Spatula Set""","""accessories"""
"""FlipMaster Long Handle Fork""","""accessories"""
"""GrillGuard Heat Resistant Gloves""","""accessories"""
"""GrillMaster Elite Tongs""","""accessories"""
"""HeatShield Premium Grill Cover""","""accessories"""
"""TempCheck Digital Thermometer""","""accessories"""
"""CleanBurn Pellets""","""consumables"""
"""FlameStarter Natural Fire Lighter""","""consumables"""
"""SeasonPro BBQ Rub Collection""","""consumables"""


## 2. Break reviews into sentences

In [4]:
sentences = split_sentences(reviews).collect()
print(f"{sentences.height} sentences, {sentences.select(KEY).n_unique()} distinct to classify")
sentences.select("review_row", "product_name", "sentence_index", "sentence_count", "sentence").head(12)

1108 sentences, 693 distinct to classify


review_row,product_name,sentence_index,sentence_count,sentence
u32,str,u32,u32,str
0,"""FireMaster Pro 3000 Gas Grill""",0,10,"""Absolutely blown away by the FireMaster Pro 3000!"""
0,"""FireMaster Pro 3000 Gas Grill""",1,10,"""Been grilling for over 20 years and this is hands down the best investment I've ever made."""
0,"""FireMaster Pro 3000 Gas Grill""",2,10,"""The temperature control is incredible - holds steady within 5 degrees of your target temp."""
0,"""FireMaster Pro 3000 Gas Grill""",3,10,"""The build quality is outstanding, heavy gauge steel that's going to last decades."""
0,"""FireMaster Pro 3000 Gas Grill""",4,10,"""Assembly was straightforward with clear instructions, took me about 3 hours working at a l…"
0,"""FireMaster Pro 3000 Gas Grill""",5,10,"""The cooking space is generous, easily fits 12 burgers or a full rack of ribs with room to …"
0,"""FireMaster Pro 3000 Gas Grill""",6,10,"""Heat distribution is perfectly even - no more hot spots burning one side while the other s…"
0,"""FireMaster Pro 3000 Gas Grill""",7,10,"""The side burner is powerful enough for serious searing, and the warming rack keeps everyth…"
0,"""FireMaster Pro 3000 Gas Grill""",8,10,"""My neighbors are already asking to borrow it!"""


## 3. Ask Jev about every sentence

Each call sends one sentence **plus the whole review** as JSON state, and asks 26 questions at once - Jev evaluates them in parallel, so extra questions barely cost latency.

| Question | Type | Why |
|---|---|---|
| `frustration` | Score, 5 levels | How frustrated is the customer? A continuous 0-4. |
| `problem_category` | Choice | Product quality, shipping, ease of use... or `none`. |
| `mentions__<product>` × 17 | Noul each | Which catalogue products the sentence refers to. One Noul per product, because a sentence can mention several. |
| `language` | Choice | Which common language it is written in. |
| `sentiment` | Choice | positive / negative / mixed / neutral. |
| `recommendation` | Choice | Recommends / warns others off / neither. |
| `churn_risk` | Noul | Returning it, refund, switching brand, won't buy again. |
| `safety_concern` | Noul | Burns, fire, gas, unsafe food - an escalation trigger. |
| `suggestion` | Noul | An explicit product improvement idea. |
| `competitor_mention` | Noul | Mentions a product outside our range. |

The star rating and demographics are deliberately **not** sent - see the sanity check below. Answers are cached in `data/output/`, so re-running only pays for new sentences.

In [5]:
questions = build_questions(catalogue.to_dicts())
print(len(questions), "questions per sentence; question set", QUESTION_SET_VERSION)
print("frustration rubric:", *[f"  {i}: {level}" for i, level in enumerate(FRUSTRATION_LEVELS)], sep="\n")

26 questions per sentence; question set 2026-09-18.3
frustration rubric:
  0: Not frustrated: positive, neutral or purely factual.
  1: Mild: a minor gripe or slight disappointment, said calmly.
  2: Clearly frustrated: annoyed or let down by a real problem.
  3: Very frustrated: angry, feels cheated, or the problem ruined the experience.
  4: Furious: hostile or emphatic language, demands a refund, or vows never to buy again.


In [6]:
answers = await classify_sentences(sentences, catalogue, cache_path=OUTPUT / "jev_sentence_answers.parquet")

answers.select(
    pl.col("jev_model").unique().alias("model"),
    pl.col("latency_ms").mean().round().alias("mean_latency_ms"),
    pl.col("input_tokens").sum().alias("input_tokens"),
    pl.col("error").is_not_null().sum().alias("errors"),
)

0 sentence(s) to classify, 693 cached.


model,mean_latency_ms,input_tokens,errors
str,f64,i64,u32
"""jev-1.13.0""",351.0,1925826,0


### Watch the wire

`transcript.log_to_stdout()` prints every exchange with Jev as JSON: the request body exactly as the SDK sent it (state, model alias, questions) and the response body exactly as the API returned it (the versioned model, every answer with its full probability distribution, token usage), plus the request ID and latency. The question set is printed in full once and elided after that - pass `full_questions=True` to see it every time. The API key travels in a header and is never printed.

The run above came from the cache, so this asks about two sentences afresh. To watch the whole run instead, call `transcript.log_to_stdout()` before it and delete `data/output/jev_sentence_answers.parquet`.

In [7]:
transcript.log_to_stdout()
await classify_sentences(sentences.head(2), catalogue)  # no cache_path, so these always call the API
transcript.silence()

2 sentence(s) to classify, 0 cached.


{
  "at": "2026-09-18T11:59:52.617+00:00",
  "sentence": {"review_key": "dd3ea70e9f3a", "sentence_index": 1},
  "request_id": "req_01a0b4632e4e7b68865f68a25dfa81a8",
  "latency_ms": 594,
  "sent": {
    "state": {
      "task": "Classify one sentence from a customer review of a barbecue product.",
      "note": "Judge what the sentence itself says. The full review is context for resolving what 'it' or 'this' refers to.",
      "sentence": "Been grilling for over 20 years and this is hands down the best investment I've ever made.",
      "sentence_position": "2 of 10",
      "review": {
        "product_reviewed": "FireMaster Pro 3000 Gas Grill",
        "product_category": "grills",
        "full_text": "Absolutely blown away by the FireMaster Pro 3000! Been grilling for over 20 years and this is hands down the best investment I've ever made. The temperature control is incredible - holds steady within 5 degrees of your target temp. The build quality is outstanding, heavy gauge steel th

{
  "at": "2026-09-18T11:59:52.646+00:00",
  "sentence": {"review_key": "dd3ea70e9f3a", "sentence_index": 0},
  "request_id": "req_01a0b4632e67769599cb65a843ba5e85",
  "latency_ms": 632,
  "sent": {
    "state": {
      "task": "Classify one sentence from a customer review of a barbecue product.",
      "note": "Judge what the sentence itself says. The full review is context for resolving what 'it' or 'this' refers to.",
      "sentence": "Absolutely blown away by the FireMaster Pro 3000!",
      "sentence_position": "1 of 10",
      "review": {
        "product_reviewed": "FireMaster Pro 3000 Gas Grill",
        "product_category": "grills",
        "full_text": "Absolutely blown away by the FireMaster Pro 3000! Been grilling for over 20 years and this is hands down the best investment I've ever made. The temperature control is incredible - holds steady within 5 degrees of your target temp. The build quality is outstanding, heavy gauge steel that's going to last decades. Assembly was 

Classified 2 in 0.6s; 0 failed.


Join the answers back onto every sentence (duplicated reviews pick up the same answers) and save the flat result.

In [8]:
classified = sentences.join(answers, on=KEY, how="left")
classified.drop("answers_json").write_parquet(OUTPUT / "sentences_classified.parquet")
classified.select(
    "product_name", "sentence", pl.col("frustration").round(2), "problem_category", "sentiment", "products_mentioned", "language"
).head(15)

product_name,sentence,frustration,problem_category,sentiment,products_mentioned,language
str,str,f64,str,str,list[str],str
"""FireMaster Pro 3000 Gas Grill""","""Absolutely blown away by the FireMaster Pro 3000!""",0.0,"""none""","""positive""","[""FireMaster Pro 3000 Gas Grill""]","""english"""
"""FireMaster Pro 3000 Gas Grill""","""Been grilling for over 20 years and this is hands down the best investment I've ever made.""",0.0,"""none""","""positive""","[""FireMaster Pro 3000 Gas Grill""]","""english"""
"""FireMaster Pro 3000 Gas Grill""","""The temperature control is incredible - holds steady within 5 degrees of your target temp.""",0.0,"""none""","""positive""","[""FireMaster Pro 3000 Gas Grill""]","""english"""
"""FireMaster Pro 3000 Gas Grill""","""The build quality is outstanding, heavy gauge steel that's going to last decades.""",0.0,"""none""","""positive""","[""FireMaster Pro 3000 Gas Grill""]","""english"""
"""FireMaster Pro 3000 Gas Grill""","""Assembly was straightforward with clear instructions, took me about 3 hours working at a l…",0.0,"""none""","""positive""","[""FireMaster Pro 3000 Gas Grill""]","""english"""
"""FireMaster Pro 3000 Gas Grill""","""The cooking space is generous, easily fits 12 burgers or a full rack of ribs with room to …",0.0,"""none""","""positive""","[""FireMaster Pro 3000 Gas Grill""]","""english"""
"""FireMaster Pro 3000 Gas Grill""","""Heat distribution is perfectly even - no more hot spots burning one side while the other s…",0.0,"""none""","""positive""","[""FireMaster Pro 3000 Gas Grill""]","""english"""
"""FireMaster Pro 3000 Gas Grill""","""The side burner is powerful enough for serious searing, and the warming rack keeps everyth…",0.0,"""none""","""positive""","[""FireMaster Pro 3000 Gas Grill""]","""english"""
"""FireMaster Pro 3000 Gas Grill""","""My neighbors are already asking to borrow it!""",0.0,"""none""","""positive""","[""FireMaster Pro 3000 Gas Grill""]","""english"""


## 4. What the answers say

### Sanity check: does frustration track the star rating?

Jev never saw the rating, so if frustration falls as stars rise, it is reading the text rather than the number.

In [9]:
by_review = review_rollup(classified.lazy())
frustration_by_rating(by_review).collect()

rating,reviews,peak_frustration,mean_frustration
i64,u32,f64,f64
1,43,3.01,2.45
2,42,1.93,1.78
3,40,1.16,0.84
4,64,0.74,0.19
5,67,0.03,0.0


### Which problems, for which products

In [10]:
problems = problems_by_product(classified.lazy()).collect()
problems.pivot("problem_category", index="product_name", values="sentences").fill_null(0).sort("product_name")

product_name,performance,build_quality,ease_of_use,general_dissatisfaction,customer_service,price_value,size_capacity,cleaning_maintenance,other,safety,shipping_delivery,assembly
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""BBQ Baron Pellet Smoker""",6,1,1,1,0,0,7,0,0,0,0,0
"""BackYard King Compact Grill""",24,24,0,17,13,12,0,5,0,0,3,0
"""ChefsPride Stainless Steel Spatula Set""",0,1,3,0,0,2,0,0,0,4,0,0
"""CleanBurn Pellets""",13,0,0,0,0,1,0,0,0,0,0,0
"""FireMaster Pro 3000 Gas Grill""",0,0,1,0,0,1,0,0,0,0,0,1
"""FlameForge Electric Indoor Grill""",28,0,0,8,0,4,0,0,0,0,0,0
"""FlameStarter Natural Fire Lighter""",14,0,0,6,0,0,0,0,4,4,0,0
"""FlipMaster Long Handle Fork""",0,4,7,0,0,4,3,4,0,0,0,0
"""GrillGuard Heat Resistant Gloves""",1,0,21,0,1,0,7,0,0,2,0,0


In [11]:
# The most frustrating problem areas, where at least 3 reviews raise them
problems.filter(pl.col("reviews") >= 3).sort("mean_frustration", descending=True).head(10)

product_name,problem_category,sentences,reviews,mean_frustration
str,str,u32,u32,f64
"""BackYard King Compact Grill""","""general_dissatisfaction""",17,12,3.31
"""TurboGrill Portable Gas""","""general_dissatisfaction""",5,3,3.13
"""BackYard King Compact Grill""","""customer_service""",13,10,3.0
"""FlameForge Electric Indoor Grill""","""general_dissatisfaction""",8,6,2.97
"""FlameForge Electric Indoor Grill""","""price_value""",4,4,2.85
"""BackYard King Compact Grill""","""price_value""",12,12,2.71
"""TempCheck Digital Thermometer""","""price_value""",4,4,2.46
"""BackYard King Compact Grill""","""performance""",24,18,2.4
"""BackYard King Compact Grill""","""cleaning_maintenance""",5,5,2.31


### Language

Every review in this sample is English, so this is a guard for when new data arrives rather than a finding.

In [12]:
classified.group_by("language").agg(pl.len(), pl.col("language_confidence").min().alias("min_confidence"))

language,len,min_confidence
str,u32,f64
"""english""",1108,1.0


### Products mentioned in reviews of *other* products

Compatibility and bundling signals - e.g. a cover reviewed alongside the grill it fits.

In [13]:
cross_mentions(classified.lazy()).collect()

product_name,also_mentions,sentence
str,str,str
"""HeatShield Premium Grill Cover""","""FireMaster Pro 3000 Gas Grill""","""Solid grill cover that fits my FireMaster perfectly."""


### Escalations: safety concerns and churn risk

In [14]:
flagged(classified.lazy(), "safety_concern").collect()

product_name,sentence,safety_concern,occurrences,frustration
str,str,f64,u32,f64
"""GrillGuard Heat Resistant Gloves""","""Claimed to handle 932°F but my hands got burned picking up a moderately hot grate.""",0.97,1,2.52
"""TurboGrill Portable Gas""","""Gas connections leak.""",0.97,2,1.51
"""GrillGuard Heat Resistant Gloves""","""False advertising and dangerous.""",0.92,1,3.24
"""ChefsPride Stainless Steel Spatula Set""","""Handles get extremely hot and become uncomfortable to use.""",0.78,1,1.98
"""ChefsPride Stainless Steel Spatula Set""","""Only complaint is the handles get quite hot during longer cooking sessions.""",0.77,2,1.0
"""BackYard King Compact Grill""","""The temperature gauge is completely inaccurate, shows 300°F when it's clearly much hotter,…",0.76,3,2.99
"""TurboGrill Portable Gas""","""Legs are wobbly and the whole thing feels like it might tip over.""",0.71,2,1.98
"""TempCheck Digital Thermometer""","""Shows 200°F when meat is clearly still raw, then jumps to 250°F instantly.""",0.71,2,1.99
"""BackYard King Compact Grill""","""Temperature control doesn't work and the whole thing feels like it's about to collapse.""",0.7,1,2.78


In [15]:
flagged(classified.lazy(), "churn_risk").collect()

product_name,sentence,churn_risk,occurrences,frustration
str,str,f64,u32,f64
"""BackYard King Compact Grill""","""I'm returning it and will never buy from this company again.""",0.97,3,4.0
"""GrillMaster Elite Tongs""","""Going back to my old set.""",0.92,3,1.48
"""GrillGuard Heat Resistant Gloves""","""Returning immediately.""",0.92,1,2.59
"""GrillMaster Elite Tongs""","""Returning immediately.""",0.88,1,2.41
"""GrillGuard Heat Resistant Gloves""","""Prefer my old leather gloves.""",0.84,2,0.93
"""WoodChips Hickory Smoking Chips""","""Find a different supplier.""",0.78,2,2.63
"""TempCheck Digital Thermometer""","""Waste of money - get a different brand.""",0.73,1,2.87
"""CleanBurn Pellets""","""Find a better brand.""",0.73,1,2.27
"""WoodChips Hickory Smoking Chips""","""Get your chips elsewhere.""",0.72,2,2.58


### Free product ideas: explicit suggestions

In [16]:
flagged(classified.lazy(), "suggestion").collect()

product_name,sentence,suggestion,occurrences,frustration
str,str,f64,u32,f64
"""HeatShield Premium Grill Cover""","""Only wish it came in different colors besides basic black.""",0.98,2,0.99
"""FlipMaster Long Handle Fork""","""Only wish the handle had better grip texture.""",0.96,2,1.0
"""BBQ Baron Pellet Smoker""","""Only complaint is the hopper could be larger for really long cooks.""",0.95,1,1.0
"""TempCheck Digital Thermometer""","""Only complaint is the probe cables could be longer for larger grills.""",0.95,1,1.0
"""TempCheck Digital Thermometer""","""Only downside is the probe cord could be longer for larger grills.""",0.95,2,1.0
"""FlipMaster Long Handle Fork""","""Sturdy construction but the handle could be more comfortable for long cooking sessions.""",0.89,2,1.0
"""BBQ Baron Pellet Smoker""","""Hopper capacity could be larger for really long cooks but overall very satisfied with perf…",0.88,2,0.83
"""FlipMaster Long Handle Fork""","""Fork does the job but handle could be more comfortable.""",0.83,1,1.0
"""ChefsPride Stainless Steel Spatula Set""","""Handles could be more comfortable but overall good value.""",0.8,1,0.95


### Competitors and previous products

In [17]:
flagged(classified.lazy(), "competitor_mention").collect()

product_name,sentence,competitor_mention,occurrences,frustration
str,str,f64,u32,f64
"""GrillGuard Heat Resistant Gloves""","""Prefer my old leather gloves.""",0.94,2,0.93
"""GrillMaster Elite Tongs""","""Going back to my old set.""",0.92,3,1.48
"""GrillMaster Elite Tongs""","""After years of using cheap ones that broke or bent, these are a revelation.""",0.8,1,0.14
"""CleanBurn Pellets""","""Doesn't impart much smoke taste compared to other brands I've tried.""",0.68,2,1.01
"""GrillGuard Heat Resistant Gloves""","""Heat protection is good but dexterity suffers.""",0.62,2,1.05
"""TempCheck Digital Thermometer""","""Waste of money - get a different brand.""",0.62,1,2.87
"""SmokeRing Premium Lump Charcoal""","""Only downside is it burns a bit faster than some other brands so you go through it quicker…",0.61,2,1.0
"""GrillGuard Heat Resistant Gloves""","""Gloves are bulky and make it hard to grip smaller items.""",0.53,2,1.28
"""SmokeRing Premium Lump Charcoal""","""Only downside is it burns faster than regular charcoal so you use more.""",0.52,1,0.99


### Low confidence: route to a person

Jev returns a confidence with every Choice. Where it could not separate the problem categories, that is a signal to send the sentence for human review rather than trust the label.

In [18]:
needs_review(classified.lazy(), min_confidence=0.6).collect()

product_name,sentence,problem_category,problem_category_confidence
str,str,str,f64
"""FlameStarter Natural Fire Lighter""","""Works but there are better natural options available.""","""other""",0.21
"""WoodChips Hickory Smoking Chips""","""Better chips available elsewhere.""","""performance""",0.23
"""TempCheck Digital Thermometer""","""App could use some improvements but overall very satisfied.""","""none""",0.3
"""BBQ Baron Pellet Smoker""","""Pellet smoker works as advertised but WiFi connectivity is unreliable.""","""performance""",0.3
"""GrillGuard Heat Resistant Gloves""","""Returning immediately.""","""customer_service""",0.31
"""WoodChips Hickory Smoking Chips""","""Wood chips are full of dirt and bark.""","""performance""",0.35
"""WoodChips Hickory Smoking Chips""","""Terrible wood chips full of dust and debris.""","""build_quality""",0.36
"""CleanBurn Pellets""","""Inconsistent sizes cause feeding problems in my smoker.""","""performance""",0.37
"""WoodChips Hickory Smoking Chips""","""Some pieces are too large while others are sawdust.""","""size_capacity""",0.38


### Most frustrated reviews

In [19]:
by_review.sort("peak_frustration", descending=True).select(
    "rating", "product_name", pl.col("peak_frustration").round(2), "problems", "churn_risk", "safety_concern", "review_text"
).unique("review_text", maintain_order=True).head(10).collect()

rating,product_name,peak_frustration,problems,churn_risk,safety_concern,review_text
i64,str,f64,list[str],bool,bool,str
1,"""BackYard King Compact Grill""",4.0,"[""build_quality"", ""cleaning_maintenance"", … ""shipping_delivery""]",true,true,"""Total waste of money! This grill arrived with multiple dents and scratches, clearly damage…"
1,"""BackYard King Compact Grill""",3.91,"[""build_quality"", ""general_dissatisfaction"", ""performance""]",false,false,"""Worst grill purchase ever made. Everything about this BackYard King is cheap and poorly de…"
1,"""BackYard King Compact Grill""",3.62,"[""customer_service"", ""general_dissatisfaction"", … ""price_value""]",false,false,"""Absolutely terrible grill with major design flaws. Temperature control doesn't work, heat …"
1,"""BackYard King Compact Grill""",3.59,"[""build_quality"", ""cleaning_maintenance"", … ""price_value""]",false,false,"""Worst grill ever made. Everything about it is cheap and poorly designed. The temperature k…"
1,"""TurboGrill Portable Gas""",3.51,"[""build_quality"", ""general_dissatisfaction"", ""safety""]",false,true,"""Portable grill is a disaster. Ignition system failed on the second use. Gas connections le…"
1,"""BackYard King Compact Grill""",3.36,"[""build_quality"", ""general_dissatisfaction"", ""performance""]",false,true,"""This grill is a complete disaster. Poor quality materials, terrible design, and non-existe…"
1,"""GrillGuard Heat Resistant Gloves""",3.24,"[""customer_service"", ""performance"", ""safety""]",true,true,"""Heat resistant gloves are a joke. Claimed to handle 932°F but my hands got burned picking …"
1,"""GrillMaster Elite Tongs""",3.17,"[""build_quality"", ""customer_service""]",true,false,"""Tongs broke on first use. Spring mechanism failed and one handle cracked. Completely unacc…"
1,"""BackYard King Compact Grill""",3.07,"[""build_quality"", ""customer_service"", ""performance""]",true,true,"""Terrible quality control. Grill arrived with uneven legs so it wobbles constantly. Burner …"
